# JICTASA-2026-018 — Corrected Evaluation Runbook (Colab)

Runs the fixed harness (`eval_harness_fixed.py`) for **8 model rows × 3 leakage-free datasets**
plus a latency micro-benchmark. Total ≈ **2–3 h on A100**, ≈ 4–5.5 h on L4.

**Before you start (one-time):**
1. Accept licenses on Hugging Face for `meta-llama/Llama-2-7b-hf` and `meta-llama/Meta-Llama-3-8B` (visit each model page while logged in).
2. Have an HF **read** token ready (Settings → Access Tokens).
3. Runtime → Change runtime type → **A100 GPU** (Colab Pro). L4 also works; avoid T4.

Every (model, dataset) run writes its own CSVs incrementally — if the session dies,
just **re-run the eval cell**: finished combinations are skipped automatically.

In [ ]:
# 1. GPU check — A100/L4 expected (T4 lacks bf16; see README fallback)
import subprocess
smi = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip()
print(smi)
assert smi, 'No GPU detected — set Runtime > Change runtime type > GPU'
if 'T4' in smi:
    print('WARNING: T4 has no bf16 — expect failures. Use A100/L4, or add --quant nf4')

In [ ]:
# 2. Pinned environment (~2 min)
%pip -q install transformers==4.44.2 peft==0.11.1 accelerate==0.33.0 \
    bitsandbytes==0.43.3 datasets==2.20.0 scikit-learn pandas tqdm sentencepiece

In [ ]:
# 3. Get the code + frozen eval sets
import os
if not os.path.isdir('FinistralAI_code'):
    !git clone https://github.com/ayansk11/FinistralAI_code
    # If the clone fails (private/stale repo): upload FinistralAI_code.zip via the
    # Files sidebar, then:  !unzip -q FinistralAI_code.zip
%cd FinistralAI_code
assert os.path.isfile('data_eval/fpb_decontam.csv'), 'data_eval/ missing — push/upload it first'
for f in ('fpb_decontam', 'fiqa', 'tfns'):
    n = sum(1 for _ in open(f'data_eval/{f}.csv')) - 1
    print(f'{f}: {n} rows')

In [ ]:
# 4. Hugging Face login (paste your READ token; needed for gated Llama repos)
from huggingface_hub import login
login()

In [ ]:
# 5. Full evaluation sweep — resumable; re-run this cell after any crash.
# 8 rows x 3 datasets. finistral_alpaca is the prompt-template ablation.
MODELS = ['finistral', 'finistral_alpaca', 'mistral_base', 'fingpt_llama2',
          'fingpt_llama3', 'fingpt_falcon', 'fingpt_bloom', 'finbert']
DATASETS = ['fpb_decontam', 'fiqa', 'tfns']
for m in MODELS:
    for d in DATASETS:
        print(f'\n########## {m} / {d} ##########')
        !python eval_harness_fixed.py --models {m} --dataset {d} \
            --quant none --seeds 0 --capture_scores \
            --batch_size 8 --max_new_tokens 8 --out_dir results_fixed

In [ ]:
# 6. Latency micro-benchmark (fills the manuscript's deployment-latency claim)
import json, time, statistics, torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import importlib.util
spec = importlib.util.spec_from_file_location('eh', 'eval_harness_fixed.py')
eh = importlib.util.module_from_spec(spec); spec.loader.exec_module(eh)

tok = AutoTokenizer.from_pretrained('mistralai/Mistral-7B-v0.1'); tok.padding_side = 'left'
if tok.pad_token is None: tok.pad_token = tok.unk_token or tok.eos_token
model = AutoModelForCausalLM.from_pretrained('mistralai/Mistral-7B-v0.1',
    torch_dtype=torch.bfloat16, device_map='auto')
model = PeftModel.from_pretrained(model, 'Ayansk11/Finistral-7B_lora'); model.eval()

import csv
sents = [r['sentence'] for r in csv.DictReader(open('data_eval/fpb_decontam.csv'))][:55]
spec_f = eh.MODELS['finistral']
times = []
for i, s in enumerate(sents):
    prompt = spec_f.template(spec_f.instruction, s)
    enc = tok(prompt, return_tensors='pt').to(model.device)
    torch.cuda.synchronize(); t0 = time.perf_counter()
    with torch.no_grad():
        model.generate(**enc, max_new_tokens=8, do_sample=False, num_beams=1,
                       pad_token_id=tok.pad_token_id)
    torch.cuda.synchronize()
    if i >= 5: times.append((time.perf_counter() - t0) * 1000)  # skip 5 warmups
gpu = torch.cuda.get_device_name(0)
res = {'gpu': gpu, 'dtype': 'bfloat16', 'batch_size': 1, 'max_new_tokens': 8,
       'decoding': 'greedy', 'n': len(times),
       'median_ms': round(statistics.median(times), 1),
       'p95_ms': round(sorted(times)[int(0.95 * len(times))], 1)}
json.dump(res, open('results_fixed/latency.json', 'w'), indent=2)
print(res)

In [ ]:
# 7. Package results for download (Files sidebar or Drive)
!zip -qr results_fixed.zip results_fixed/
print('Download results_fixed.zip from the Files sidebar, then drop it into the repo root locally.')
# Optional: copy to Drive instead —
# from google.colab import drive; drive.mount('/content/drive')
# !cp results_fixed.zip /content/drive/MyDrive/